# Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/drive/MyDrive/2dod/2dod/data.yaml",
    epochs=200,
    imgsz=640,
    device=0,
    project="yolov8s_carlagear",
    name="train"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/drive/MyDrive/2dod/2dod/data.yaml",
    epochs=200,
    imgsz=640,
    device=0,
    batch=16,       # T4 safe for YOLOv8s
    workers=2,      # Colab stable
    cache=True,     # faster training
    amp=True,       # mixed precision on T4
    project="/content/drive/MyDrive/yolov8s_carlagear",
    name="train"
)

In [ ]:
import os

weights = '/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt'
size = os.path.getsize(weights) / (1024*1024)

print('File exists:', os.path.exists(weights))
print('File size:', round(size, 1), 'MB')

Yolov5n

In [ ]:
!pip install -U ultralytics

from ultralytics import YOLO

# Load YOLOv5 nano
model = YOLO("yolov5n.pt")

# Train
results = model.train(
    data="/content/drive/MyDrive/2dod/2dod/data.yaml",
    epochs=200,
    imgsz=640,
    device=0,
    batch=16,        # usually safe on T4
    workers=2,       # stable for Colab
    cache=True,
    amp=True,
    project="/content/drive/MyDrive/yolov5n_carlagear",
    name="train"
)

MobileNet-SSD training



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torchvision -q

import os, csv, glob, torch, torchvision
import torch.nn as nn
from torchvision import transforms
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from functools import partial
from PIL import Image

print('GPU:', torch.cuda.get_device_name(0))

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/2dod/2dod'
TRAIN_OUT   = '/content/drive/MyDrive/mobilenet_ssd_carlagear'
SAVE_PATH   = f'{TRAIN_OUT}/ssdlite_best.pth'
CKPT_DIR    = f'{TRAIN_OUT}/checkpoints'
LOG_PATH    = f'{TRAIN_OUT}/training_log.csv'

# ── YOUR 6 CLASSES + background ───────────────────────────────────────────────
# YOLO class 0..5 → model label 1..6 (0 = background)
NUM_CLASSES = 7        # 6 classes + background
NUM_EPOCHS  = 200
BATCH_SIZE  = 16

os.makedirs(TRAIN_OUT, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)
print('✅ Output dir ready:', TRAIN_OUT)

# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── YOLO-format dataset class ─────────────────────────────────────────────────
class YOLODetectionDataset(Dataset):
    """Reads YOLO .txt labels alongside images."""

    def __init__(self, img_dir, label_dir, transform=None):
        self.transform = transform
        self.samples = []
        for img_path in sorted(glob.glob(f'{img_dir}/*.png') +
                               glob.glob(f'{img_dir}/*.jpg')):
            stem = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(label_dir, stem + '.txt')
            self.samples.append((img_path, lbl_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size

        boxes, labels = [], []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts[:5])
                    # YOLO → xyxy (pixel)
                    x1 = (cx - bw / 2) * W
                    y1 = (cy - bh / 2) * H
                    x2 = (cx + bw / 2) * W
                    y2 = (cy + bh / 2) * H
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(W, x2), min(H, y2)
                    if x2 > x1 and y2 > y1:
                        boxes.append([x1, y1, x2, y2])
                        labels.append(int(cls) + 1)  # +1 for background=0

        if self.transform:
            img = self.transform(img)

        target = {
            'boxes':  torch.tensor(boxes,  dtype=torch.float32).reshape(-1, 4),
            'labels': torch.tensor(labels, dtype=torch.int64),
        }
        # SSD needs boxes in [0,1] relative coords after resize to 320x320
        if len(boxes) > 0:
            target['boxes'][:, [0, 2]] /= W
            target['boxes'][:, [1, 3]] /= H

        return img, target


def make_datasets(split, transform):
    datasets = []
    for i in range(1, 10):
        scenario = f'billboard0{i}'
        img_dir  = f'{DRIVE_BASE}/{scenario}/images/{split}'
        lbl_dir  = f'{DRIVE_BASE}/{scenario}/labels/{split}'
        if os.path.exists(img_dir) and os.path.exists(lbl_dir):
            datasets.append(YOLODetectionDataset(img_dir, lbl_dir, transform))
        else:
            print(f'WARNING: missing {scenario}/{split}')
    return ConcatDataset(datasets)

train_dataset = make_datasets('train', train_transform)
val_dataset   = make_datasets('val',   val_transform)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)
print('Dataloaders ready.')


In [ ]:
# ── CELL: Sanity check before training ───────────────────────
print('=== 1. Dataset ===')
sample_img, sample_target = train_dataset[0]
print(f'  Image shape : {sample_img.shape}')   # expect [3, 320, 320]
print(f'  Boxes shape : {sample_target["boxes"].shape}')
print(f'  Labels      : {sample_target["labels"]}')
print(f'  Label range : {sample_target["labels"].min().item()} – '
      f'{sample_target["labels"].max().item()}')  # expect 1–6, never 0 or >6

print('\n=== 2. Dataloader (one batch) ===')
images, targets = next(iter(train_loader))
print(f'  Batch size  : {len(images)}')
print(f'  Image type  : {type(images[0])}')
print(f'  Image shape : {images[0].shape}')   # expect [3, 320, 320]

print('\n=== 3. Forward pass (train mode = loss computation) ===')
model_ssd.train()
images_gpu  = [img.to(device) for img in images]
targets_gpu = [{k: v.to(device) for k, v in t.items()} for t in targets]
with torch.no_grad():
    loss_dict = model_ssd(images_gpu, targets_gpu)
print(f'  Loss keys   : {list(loss_dict.keys())}')
print(f'  Loss values : { {k: round(v.item(),4) for k,v in loss_dict.items()} }')
total = sum(loss_dict.values()).item()
print(f'  Total loss  : {total:.4f}')   # should be a finite positive number

print('\n=== 4. Forward pass (eval mode = inference) ===')
model_ssd.eval()
with torch.no_grad():
    preds = model_ssd(images_gpu[:1])
print(f'  Boxes  : {preds[0]["boxes"].shape}')
print(f'  Labels : {preds[0]["labels"][:5]}')
print(f'  Scores : {preds[0]["scores"][:5]}')

print('\n=== 5. Backward pass (one gradient step) ===')
model_ssd.train()
loss_dict = model_ssd(images_gpu, targets_gpu)
loss = sum(loss_dict.values())
loss.backward()
print(f'  Backward OK — loss: {loss.item():.4f}')
optimizer.zero_grad()

print('\n✅ All checks passed — safe to start training')

In [ ]:

# ── Model ─────────────────────────────────────────────────────────────────────
model_ssd = torchvision.models.detection.ssdlite320_mobilenet_v3_large(
    weights='DEFAULT'
)

in_channels = []
for module in model_ssd.head.classification_head.module_list:
    for layer in module:
        if hasattr(layer, 'in_channels'):
            in_channels.append(layer.in_channels)
            break

num_anchors = model_ssd.anchor_generator.num_anchors_per_location()
print('in_channels:', in_channels)
print('num_anchors:', num_anchors)

model_ssd.head.classification_head = SSDLiteClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=NUM_CLASSES,   # 7 = 6 classes + background
    norm_layer=partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_ssd = model_ssd.to(device)
print('Model ready on:', device)

# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model_ssd.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_loss     = float('inf')
epochs_no_improve = 0
patience          = 50
log_rows          = []

for epoch in range(NUM_EPOCHS):
    # ── Train ──
    model_ssd.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images  = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model_ssd(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ssd.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ──
    model_ssd.train()   # SSD needs train mode to compute losses
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images  = list(img.to(device) for img in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model_ssd(images, targets)
            val_loss += sum(loss_dict.values()).item()
    val_loss /= len(val_loader)

    scheduler.step()
    log_rows.append((epoch, train_loss, val_loss))

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.2e}')

    # Save checkpoint every 20 epochs
    if epoch % 20 == 0:
        ckpt = f'{CKPT_DIR}/ckpt_epoch{epoch:03d}.pth'
        torch.save({'epoch': epoch,
                    'model_state_dict': model_ssd.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss,
                    'val_loss':   val_loss}, ckpt)
        print(f'  Checkpoint saved: {ckpt}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model_ssd.state_dict(), SAVE_PATH)
        print(f'  Best saved at epoch {epoch} | Val: {val_loss:.4f}')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

print('Training complete. Best val loss:', best_val_loss)

# Save log
with open(LOG_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'val_loss'])
    writer.writerows(log_rows)
print('Log saved to:', LOG_PATH)

In [ ]:
import os
project_root = '/content/drive/MyDrive/mobilenet_ssd_carlagear'

for root, dirs, files in os.walk(project_root):
    for f in files:
        if f.endswith('.py'):
            print(os.path.join(root, f))

In [ ]:
import inspect

print("=== FUNCTIONS / CLASSES / OBJECTS IN MEMORY ===")
for name, obj in list(globals().items()):
    try:
        if inspect.isfunction(obj) or inspect.isclass(obj):
            print("FUNC/CLASS:", name)
    except:
        pass

for name, obj in list(globals().items()):
    if name in ["model", "net", "criterion", "train_loader", "val_loader", "optimizer"]:
        print("OBJECT:", name, type(obj))

In [ ]:
import inspect

print(inspect.signature(YOLODetectionDataset.__init__))

In [ ]:
# Clone the most widely used PyTorch implementation
!git clone https://github.com/zylo117/Yet-Another-EfficientDet-Pytorch
%cd Yet-Another-EfficientDet-Pytorch
!pip install pycocotools numpy opencv-python tqdm tensorboard \
             tensorboardX pyyaml webcolors
!pip install torch torchvision --upgrade

In [ ]:
import os, json
from pathlib import Path
from PIL import Image

# YOLO class ID → name (your 80-class COCO setup)
# We only care about the classes that appear in your data
COCO_NAMES = {
    0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle',
    5: 'bus', 7: 'truck', 9: 'traffic light', 11: 'stop sign'
}

def yolo_to_coco(img_dir, label_dir, output_json):
    images = []
    annotations = []
    ann_id = 1

    # Build category list from classes actually present
    present_classes = set()
    for txt in Path(label_dir).glob('*.txt'):
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                present_classes.add(int(float(parts[0])))

    # Map class IDs to contiguous 1-indexed COCO IDs
    sorted_classes = sorted(present_classes)
    class_map = {c: i+1 for i, c in enumerate(sorted_classes)}
    categories = [
        {"id": class_map[c], "name": COCO_NAMES.get(c, str(c)), "supercategory": "object"}
        for c in sorted_classes
    ]

    img_paths = sorted(Path(img_dir).glob('*.png')) + \
                sorted(Path(img_dir).glob('*.jpg'))

    for img_id, img_path in enumerate(img_paths, 1):
        img = Image.open(img_path)
        W, H = img.size

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": W,
            "height": H
        })

        txt_path = Path(label_dir) / f"{img_path.stem}.txt"
        if not txt_path.exists():
            continue

        for line in txt_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            c = int(float(parts[0]))
            if c not in class_map:
                continue
            x, y, w, h = map(float, parts[1:5])
            x1 = (x - w/2) * W
            y1 = (y - h/2) * H
            bw = w * W
            bh = h * H

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": class_map[c],
                "bbox": [x1, y1, bw, bh],
                "area": bw * bh,
                "iscrowd": 0
            })
            ann_id += 1

    coco = {"images": images, "annotations": annotations, "categories": categories}
    Path(output_json).parent.mkdir(parents=True, exist_ok=True)
    Path(output_json).write_text(json.dumps(coco, indent=2))
    print(f"Saved {len(images)} images, {len(annotations)} annotations → {output_json}")
    return categories, class_map

# Convert all 9 billboards
base = '/content/drive/MyDrive/2dod/2dod'
coco_out = '/content/efficientdet_data'

for bb in [f'billboard0{i}' for i in range(1, 10)]:
    for split in ['train', 'val']:
        img_dir   = f'{base}/{bb}/images/{split}'
        label_dir = f'{base}/{bb}/labels/{split}'
        if not os.path.exists(img_dir):
            continue
        out_json = f'{coco_out}/annotations/{bb}_{split}.json'
        yolo_to_coco(img_dir, label_dir, out_json)
        print(f'Done: {bb}/{split}')

In [ ]:
import os, shutil
from pathlib import Path

coco_out = '/content/efficientdet_data'

# EfficientDet expects:
# datasets/carlagear/train/ (images)
# datasets/carlagear/val/   (images)
# datasets/carlagear/annotations/instances_train.json
# datasets/carlagear/annotations/instances_val.json

base = '/content/drive/MyDrive/2dod/2dod'
ds_root = Path('datasets/carlagear')

for split in ['train', 'val']:
    dst = ds_root / split
    dst.mkdir(parents=True, exist_ok=True)

# Merge all billboard annotations into one train and one val JSON
def merge_coco_jsons(json_paths, output_path):
    merged = {"images": [], "annotations": [], "categories": None}
    img_id_offset = 0
    ann_id_offset = 0
    for jp in json_paths:
        if not Path(jp).exists():
            continue
        d = json.loads(Path(jp).read_text())
        if merged["categories"] is None:
            merged["categories"] = d["categories"]
        for img in d["images"]:
            new_img = dict(img)
            new_img["id"] = img["id"] + img_id_offset
            merged["images"].append(new_img)
        for ann in d["annotations"]:
            new_ann = dict(ann)
            new_ann["id"] = ann["id"] + ann_id_offset
            new_ann["image_id"] = ann["image_id"] + img_id_offset
            merged["annotations"].append(new_ann)
        if d["images"]:
            img_id_offset += max(img["id"] for img in d["images"]) + 1
        if d["annotations"]:
            ann_id_offset += max(ann["id"] for ann in d["annotations"]) + 1
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    Path(output_path).write_text(json.dumps(merged, indent=2))
    print(f'Merged {len(json_paths)} files → {len(merged["images"])} images → {output_path}')

ann_dir = Path('/content/efficientdet_data/annotations')
merge_coco_jsons(
    [ann_dir / f'billboard0{i}_train.json' for i in range(1,10)],
    'datasets/carlagear/annotations/instances_train.json'
)
merge_coco_jsons(
    [ann_dir / f'billboard0{i}_val.json' for i in range(1,10)],
    'datasets/carlagear/annotations/instances_val.json'
)

# Copy images
for split in ['train', 'val']:
    for i in range(1, 10):
        src = Path(f'{base}/billboard0{i}/images/{split}')
        dst = Path(f'datasets/carlagear/{split}')
        if not src.exists():
            continue
        for f in src.iterdir():
            if f.suffix.lower() in {'.jpg', '.png'}:
                shutil.copy2(f, dst / f.name)
    print(f'Copied {split} images')

In [ ]:
import yaml
from pathlib import Path
import json

# Get classes from merged annotation
sample = json.loads(Path('datasets/carlagear/annotations/instances_train.json').read_text())
class_names = [c['name'] for c in sorted(sample['categories'], key=lambda x: x['id'])]
print(f'Classes: {class_names}')

project_cfg = {
    'project_name': 'carlagear',
    'train_set': 'train',
    'val_set': 'val',
    'num_gpus': 1,
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
    'obj_list': class_names
}

Path('projects/carlagear.yml').parent.mkdir(exist_ok=True)
Path('projects/carlagear.yml').write_text(yaml.dump(project_cfg))
print('Written to:', os.path.abspath('projects/carlagear.yml'))
print(Path('projects/carlagear.yml').read_text())

In [ ]:
import json
from pathlib import Path
from collections import Counter

# Check what classes actually appear across ALL billboard train labels
base = '/content/drive/MyDrive/2dod/2dod'
class_counts = Counter()

for i in range(1, 10):
    label_dir = Path(f'{base}/billboard0{i}/labels/train')
    if not label_dir.exists():
        continue
    for txt in label_dir.glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                class_counts[int(float(parts[0]))] += 1

print('Class distribution across all billboard train labels:')
for cls, count in sorted(class_counts.items()):
    print(f'  COCO class {cls}: {count} instances')

NanoDet-Plus


In [ ]:
import os, json, shutil
from pathlib import Path
from PIL import Image

# Fixed global class map — same for ALL billboards
COCO_NAMES = {0: 'person', 2: 'car', 3: 'motorcycle', 7: 'truck', 9: 'traffic light', 11: 'stop sign'}
PRESENT_CLASSES = sorted(COCO_NAMES.keys())  # [0, 2, 3, 7, 9, 11]
CLASS_MAP = {c: i+1 for i, c in enumerate(PRESENT_CLASSES)}
CATEGORIES = [{"id": CLASS_MAP[c], "name": COCO_NAMES[c], "supercategory": "object"} for c in PRESENT_CLASSES]
print('Class map:', CLASS_MAP)

base = '/content/drive/MyDrive/2dod/2dod'

for split in ['train', 'val']:
    all_images, all_annotations = [], []
    img_id, ann_id = 1, 1

    for i in range(1, 10):
        bb = f'billboard0{i}'
        img_dir   = Path(f'{base}/{bb}/images/{split}')
        label_dir = Path(f'{base}/{bb}/labels/{split}')
        if not img_dir.exists():
            continue

        for img_path in sorted(img_dir.glob('*.png')) + sorted(img_dir.glob('*.jpg')):
            img = Image.open(img_path)
            W, H = img.size
            new_name = f'{bb}_{img_path.name}'
            all_images.append({"id": img_id, "file_name": new_name, "width": W, "height": H})

            txt_path = label_dir / f"{img_path.stem}.txt"
            if txt_path.exists():
                for line in txt_path.read_text().strip().splitlines():
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    c = int(float(parts[0]))
                    if c not in CLASS_MAP:
                        continue
                    x, y, w, h = map(float, parts[1:5])
                    x1, y1 = (x - w/2)*W, (y - h/2)*H
                    all_annotations.append({
                        "id": ann_id, "image_id": img_id,
                        "category_id": CLASS_MAP[c],
                        "bbox": [x1, y1, w*W, h*H],
                        "area": w*W * h*H, "iscrowd": 0
                    })
                    ann_id += 1
            img_id += 1

    out = f'/content/drive/MyDrive/efficientdet_carlagear/annotations/instances_{split}.json'
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    Path(out).write_text(json.dumps({"images": all_images, "annotations": all_annotations, "categories": CATEGORIES}, indent=2))
    print(f'{split}: {len(all_images)} images, {len(all_annotations)} annotations → {out}')


In [ ]:
import json
from pathlib import Path
from collections import Counter

d = json.loads(Path('/content/drive/MyDrive/efficientdet_carlagear/annotations/instances_train.json').read_text())

print('Categories:', [(c['id'], c['name']) for c in d['categories']])
print('Total images:', len(d['images']))
print('Total annotations:', len(d['annotations']))

cat_map = {c['id']: c['name'] for c in d['categories']}
dist = Counter(ann['category_id'] for ann in d['annotations'])
print('\nAnnotation distribution:')
for cid, count in sorted(dist.items()):
    print(f'  {cid} ({cat_map[cid]}): {count}')

In [ ]:
# Step 1 — Clone and install
!git clone https://github.com/RangiLyu/nanodet
%cd nanodet
!pip install -r requirements/train.txt -q
!pip install -e . -q

In [ ]:
!pip install -q -r /content/nanodet/requirements.txt --ignore-requires-python \
  --extra-index-url https://pypi.org/simple/ \
  --constraint /dev/null \
  2>&1 | grep -v "torch"

# Install nanodet without checking torch version
!cd /content/nanodet && pip install -e . --no-deps -q
print('Done')

In [ ]:
import sys
sys.path.insert(0, '/content/nanodet')
import nanodet
print('NanoDet imported successfully')

In [ ]:
import shutil, os
from pathlib import Path

os.makedirs('/content/nanodet/data/carlagear/annotations', exist_ok=True)
shutil.copy(
    '/content/drive/MyDrive/efficientdet_carlagear/annotations/instances_train.json',
    '/content/nanodet/data/carlagear/annotations/instances_train.json'
)
shutil.copy(
    '/content/drive/MyDrive/efficientdet_carlagear/annotations/instances_val.json',
    '/content/nanodet/data/carlagear/annotations/instances_val.json'
)
print('Annotations copied ✅')

base = '/content/drive/MyDrive/2dod/2dod'
for split in ['train', 'val']:
    dst = Path(f'/content/nanodet/data/carlagear/{split}')
    dst.mkdir(parents=True, exist_ok=True)
    for i in range(1, 10):
        bb = f'billboard0{i}'
        src = Path(f'{base}/{bb}/images/{split}')
        if not src.exists():
            continue
        for f in src.iterdir():
            if f.suffix.lower() in {'.jpg', '.png'}:
                shutil.copy2(f, dst / f'{bb}_{f.name}')
    print(f'Copied {split} images ✅')

In [ ]:
with open('/content/nanodet/config/carlagear.yml', 'w') as f:
    f.write("""
save_dir: /content/drive/MyDrive/nanodet_carlagear
model:
  arch:
    name: NanoDetPlus
    detach_epoch: 10
    backbone:
      name: ShuffleNetV2
      model_size: 0.5x
      out_stages: [2,3,4]
      activation: LeakyReLU
    fpn:
      name: GhostPAN
      in_channels: [48, 96, 192]
      out_channels: 96
      kernel_size: 5
      num_extra_level: 1
      use_depthwise: True
      activation: LeakyReLU
    head:
      name: NanoDetPlusHead
      num_classes: 6
      input_channel: 96
      feat_channels: 96
      stacked_convs: 2
      kernel_size: 5
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
      norm_cfg:
        type: BN
      loss:
        loss_qfl:
          name: QualityFocalLoss
          use_sigmoid: True
          beta: 2.0
          loss_weight: 1.0
        loss_dfl:
          name: DistributionFocalLoss
          loss_weight: 0.25
        loss_bbox:
          name: GIoULoss
          loss_weight: 2.0
    aux_head:
      name: SimpleConvHead
      num_classes: 6
      input_channel: 192
      feat_channels: 192
      stacked_convs: 4
      activation: LeakyReLU
      reg_max: 7

data:
  train:
    name: CocoDataset
    img_path: data/carlagear/train
    ann_path: data/carlagear/annotations/instances_train.json
    input_size: [320,320]
    keep_ratio: False
    pipeline:
      perspective: 0.0
      scale: [0.6, 1.4]
      stretch: [[1,1],[1,1]]
      rotation: 0
      shear: 0
      translate: 0.2
      flip: 0.5
      brightness: 0.2
      contrast: [0.6, 1.4]
      saturation: [0.5, 1.2]
      normalize: [[103.53,116.28,123.675],[57.375,57.12,58.395]]
  val:
    name: CocoDataset
    img_path: data/carlagear/val
    ann_path: data/carlagear/annotations/instances_val.json
    input_size: [320,320]
    keep_ratio: False
    pipeline:
      normalize: [[103.53,116.28,123.675],[57.375,57.12,58.395]]

device:
  gpu_ids: [0]
  workers_per_gpu: 2
  batchsize_per_gpu: 32

schedule:
  optimizer:
    name: AdamW
    lr: 0.001
    weight_decay: 0.05
  warmup:
    name: linear
    steps: 500
    ratio: 0.0001
  total_epochs: 100
  lr_schedule:
    name: CosineAnnealingLR
    T_max: 100
    eta_min: 0.00005
  val_intervals: 5

grad_clip: 35
evaluator:
  name: CocoDetectionEvaluator
  save_key: mAP

log:
  interval: 10
""")
print('Config written ✅')

In [ ]:
import os
os.chdir('/content/nanodet')

# Try alternative download URLs
!wget -q --show-progress "https://github.com/RangiLyu/nanodet/releases/download/v1.0.0-alpha-1/nanodet-plus-m_320.pth" -O nanodet-plus-m_320.pth || \
 wget -q --show-progress "https://github.com/RangiLyu/nanodet/releases/download/v1.0.0/nanodet-plus-m_320.pth" -O nanodet-plus-m_320.pth

print('Exists:', os.path.exists('nanodet-plus-m_320.pth'))
if os.path.exists('nanodet-plus-m_320.pth'):
    print('Size:', os.path.getsize('nanodet-plus-m_320.pth') / 1e6, 'MB')

In [ ]:
import os
files = os.listdir('/content/drive/MyDrive/')
# Show files that might be the weights
for f in sorted(files):
    if '.pth' in f.lower() or 'nano' in f.lower():
        print(f)

In [ ]:
!ls -la /content/nanodet/*.pth 2>/dev/null || echo "No .pth files in nanodet folder"

In [ ]:
!pip install -q pytorch-lightning==1.9.5
!python /content/nanodet/tools/train.py /content/nanodet/config/carlagear.yml

In [ ]:
from pathlib import Path

# Update to 200 epochs
content = Path('/content/nanodet/config/carlagear.yml').read_text()
content = content.replace('total_epochs: 100', 'total_epochs: 200')
content = content.replace('T_max: 100', 'T_max: 200')
Path('/content/nanodet/config/carlagear.yml').write_text(content)
print('Updated to 200 epochs')

In [ ]:
# Find the latest checkpoint
import os
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/nanodet_carlagear')
ckpts = sorted(ckpt_dir.rglob('*.ckpt'))
if ckpts:
    latest = str(ckpts[-1])
    print('Resuming from:', latest)
else:
    latest = None
    print('No checkpoint found — starting fresh')

In [ ]:
from pathlib import Path

# Fix 1 — torch._six patch
collate = Path('/content/nanodet/nanodet/data/collate.py')
collate.write_text(collate.read_text().replace(
    'from torch._six import string_classes',
    'string_classes = (str,)'
))
print('Patched torch._six ✅')

# Fix 2 — rewrite config
Path('/content/nanodet/config/carlagear.yml').write_text("""
save_dir: /content/drive/MyDrive/nanodet_carlagear
class_names: [person, car, motorcycle, truck, traffic light, stop sign]
model:
  arch:
    name: NanoDetPlus
    detach_epoch: 10
    backbone:
      name: ShuffleNetV2
      model_size: 0.5x
      out_stages: [2,3,4]
      activation: LeakyReLU
    fpn:
      name: GhostPAN
      in_channels: [48, 96, 192]
      out_channels: 96
      kernel_size: 5
      num_extra_level: 0
      use_depthwise: True
      activation: LeakyReLU
    head:
      name: NanoDetPlusHead
      num_classes: 6
      input_channel: 96
      feat_channels: 96
      stacked_convs: 2
      kernel_size: 5
      strides: [8, 16, 32]
      activation: LeakyReLU
      reg_max: 7
      norm_cfg:
        type: BN
      loss:
        loss_qfl:
          name: QualityFocalLoss
          use_sigmoid: True
          beta: 2.0
          loss_weight: 1.0
        loss_dfl:
          name: DistributionFocalLoss
          loss_weight: 0.25
        loss_bbox:
          name: GIoULoss
          loss_weight: 2.0
    aux_head:
      name: SimpleConvHead
      num_classes: 6
      input_channel: 192
      feat_channels: 192
      stacked_convs: 4
      activation: LeakyReLU
      reg_max: 7

data:
  train:
    name: CocoDataset
    img_path: data/carlagear/train
    ann_path: data/carlagear/annotations/instances_train.json
    input_size: [320,320]
    keep_ratio: False
    pipeline:
      perspective: 0.0
      scale: [0.6, 1.4]
      stretch: [[1,1],[1,1]]
      rotation: 0
      shear: 0
      translate: 0.2
      flip: 0.5
      brightness: 0.2
      contrast: [0.6, 1.4]
      saturation: [0.5, 1.2]
      normalize: [[103.53,116.28,123.675],[57.375,57.12,58.395]]
  val:
    name: CocoDataset
    img_path: data/carlagear/val
    ann_path: data/carlagear/annotations/instances_val.json
    input_size: [320,320]
    keep_ratio: False
    pipeline:
      normalize: [[103.53,116.28,123.675],[57.375,57.12,58.395]]

device:
  gpu_ids: [0]
  workers_per_gpu: 2
  batchsize_per_gpu: 32

schedule:
  optimizer:
    name: AdamW
    lr: 0.001
    weight_decay: 0.05
  warmup:
    name: linear
    steps: 500
    ratio: 0.0001
  total_epochs: 200
  lr_schedule:
    name: CosineAnnealingLR
    T_max: 200
    eta_min: 0.00005
  val_intervals: 5

grad_clip: 35
evaluator:
  name: CocoDetectionEvaluator
  save_key: mAP

log:
  interval: 10
""")
print('Config written ✅')

# Train
!python /content/nanodet/tools/train.py /content/nanodet/config/carlagear.yml